# 06 — Final Comparison and Validation

This notebook uses the saved schedules and source data from the earlier project notebooks. It does **not** rerun CP-SAT and does not overwrite any schedule files.

Expected location: `src/06_final_comparison_and_validation.ipynb`. Run the notebook with the project `.venv` kernel.

## 1. Imports and project paths

In [1]:
from pathlib import Path
import pickle
import pandas as pd
import numpy as np
import networkx as nx
from IPython.display import display

candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((p.resolve() for p in candidates if (p / "data" / "processed").exists()), Path.cwd().resolve())
PROCESSED = PROJECT_ROOT / "data" / "processed"
RAW = PROJECT_ROOT / "data" / "raw" / "Nott"
print("Project root:", PROJECT_ROOT)
for path in [PROCESSED / "conflict_graph.gpickle", PROCESSED / "adaptive_v3_schedule.csv", PROCESSED / "cp_sat_schedule.csv", RAW / "enrolements"]:
    print(("FOUND " if path.exists() else "MISSING ") + str(path))

Project root: C:\Users\Nethmi\Documents\GitHub\adaptive-exam-timetabling
FOUND C:\Users\Nethmi\Documents\GitHub\adaptive-exam-timetabling\data\processed\conflict_graph.gpickle
FOUND C:\Users\Nethmi\Documents\GitHub\adaptive-exam-timetabling\data\processed\adaptive_v3_schedule.csv
FOUND C:\Users\Nethmi\Documents\GitHub\adaptive-exam-timetabling\data\processed\cp_sat_schedule.csv
FOUND C:\Users\Nethmi\Documents\GitHub\adaptive-exam-timetabling\data\raw\Nott\enrolements


## 2. Load graph and saved schedules

In [2]:
graph_path = PROCESSED / "conflict_graph.gpickle"
with open(graph_path, "rb") as f:
    G = pickle.load(f)

def load_schedule(path):
    df = pd.read_csv(path)
    required = {"exam_id", "period_index"}
    if not required.issubset(df.columns):
        raise ValueError(f"{path.name} needs columns {sorted(required)}; found {df.columns.tolist()}")
    df = df.copy()
    df["exam_id"] = df["exam_id"].astype(str).str.strip()
    df["period_index"] = pd.to_numeric(df["period_index"], errors="raise").astype(int)
    if df["exam_id"].duplicated().any():
        raise ValueError(f"Duplicate exam IDs in {path.name}")
    return df

adaptive_v3 = load_schedule(PROCESSED / "adaptive_v3_schedule.csv")
cp_sat = load_schedule(PROCESSED / "cp_sat_schedule.csv")
print(f"Conflict graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
display(pd.DataFrame([
    {"Algorithm":"Adaptive v3", "Rows":len(adaptive_v3), "Unique exams":adaptive_v3.exam_id.nunique(), "Periods":adaptive_v3.period_index.nunique()},
    {"Algorithm":"CP-SAT", "Rows":len(cp_sat), "Unique exams":cp_sat.exam_id.nunique(), "Periods":cp_sat.period_index.nunique()}
]))

Conflict graph: 800 nodes, 9990 edges


,Algorithm,Rows,Unique exams,Periods
0,Adaptive v3,800,800,17
1,CP-SAT,800,800,16


## 3. Validate schedule completeness and graph conflicts

In [3]:
graph_exams = {str(x).strip() for x in G.nodes()}
def validate_schedule(schedule, label):
    exam_period = dict(zip(schedule.exam_id, schedule.period_index))
    scheduled = set(exam_period)
    missing = graph_exams - scheduled
    extra = scheduled - graph_exams
    conflicts = [(str(u), str(v)) for u, v in G.edges()
                 if str(u).strip() in exam_period and str(v).strip() in exam_period
                 and exam_period[str(u).strip()] == exam_period[str(v).strip()]]
    return {"Algorithm":label, "Graph exams":len(graph_exams), "Rows":len(schedule),
            "Unique exams":schedule.exam_id.nunique(), "Missing exams":len(missing),
            "Extra exams":len(extra), "Periods used":schedule.period_index.nunique(),
            "Graph conflict violations":len(conflicts),
            "Graph-feasible":not missing and not extra and not conflicts}, conflicts

adaptive_check, adaptive_conflicts = validate_schedule(adaptive_v3, "Adaptive v3")
cp_check, cp_conflicts = validate_schedule(cp_sat, "CP-SAT")
validation_df = pd.DataFrame([adaptive_check, cp_check])
display(validation_df)
if cp_conflicts:
    print("First CP-SAT conflicts:", cp_conflicts[:10])

,Algorithm,Graph exams,Rows,Unique exams,Missing exams,Extra exams,Periods used,Graph conflict violations,Graph-feasible
0,Adaptive v3,800,800,800,0,0,17,0,True
1,CP-SAT,800,800,800,0,0,16,0,True


## 4. Inspect the original enrolment data

The source file has no extension. This cell prints the first lines and a whitespace-separated preview so the student-ID and exam-ID columns can be confirmed before calculating a student-based penalty. Do not assume the first two columns are correct without checking this preview.

In [4]:
enrol_path = RAW / "enrolements"
if not enrol_path.exists():
    raise FileNotFoundError(enrol_path)
lines = [x for x in enrol_path.read_text(encoding="utf-8", errors="replace").splitlines() if x.strip()]
print("Non-empty lines:", len(lines))
for line in lines[:10]:
    print(repr(line))
enrol_preview = pd.read_csv(enrol_path, sep=r"\s+", header=None, engine="python", nrows=8, dtype=str)
display(enrol_preview)
print("Preview columns:", enrol_preview.shape[1])

Non-empty lines: 33997
'A890186790 R13001E1'
'A890186790 R13006E1'
'A890186790 R13016E1'
'A890186790 R13021E1'
'A890186790 R13022E1'
'A891097581 R23102E1'
'A891097581 R23107E1'
'A891097581 R23111E1'
'A891097581 R23117E1'
'A891097581 R23209E1'


,0,1
0,A890186790,R13001E1
1,A890186790,R13006E1
2,A890186790,R13016E1
3,A890186790,R13021E1
4,A890186790,R13022E1
5,A891097581,R23102E1
6,A891097581,R23107E1
7,A891097581,R23111E1


Preview columns: 2


## 5. Parse student–exam pairs

**Check the preview above first.** The default below assumes a whitespace-delimited file with no header and the first two columns representing student ID and exam ID. If the file shows a different order or includes a header, edit `STUDENT_COL_INDEX` and `EXAM_COL_INDEX` or the parser before continuing.

In [5]:
enrol_raw = pd.read_csv(enrol_path, sep=r"\s+", header=None, engine="python", dtype=str)
if enrol_raw.shape[1] < 2:
    raise ValueError("Could not parse at least two columns. Inspect Section 4 and adjust the parser.")
STUDENT_COL_INDEX = 0
EXAM_COL_INDEX = 1
enrolments = enrol_raw.iloc[:, [STUDENT_COL_INDEX, EXAM_COL_INDEX]].copy()
enrolments.columns = ["student_id", "exam_id"]
enrolments = enrolments.dropna()
enrolments["student_id"] = enrolments.student_id.astype(str).str.strip()
enrolments["exam_id"] = enrolments.exam_id.astype(str).str.strip()
enrolments = enrolments.drop_duplicates()
print("Parsed pairs:", len(enrolments))
print("Unique students:", enrolments.student_id.nunique())
print("Unique exam IDs:", enrolments.exam_id.nunique())
display(enrolments.head())

Parsed pairs: 33997
Unique students: 7896
Unique exam IDs: 800


,student_id,exam_id
0,A890186790,R13001E1
1,A890186790,R13006E1
2,A890186790,R13016E1
3,A890186790,R13021E1
4,A890186790,R13022E1


## 6. Calculate the same-day adjacent-period penalty

This implementation counts consecutive exams for the same student when their scheduled periods are adjacent (period indices differ by one) and the date is the same. This is only comparable with the earlier results if the earlier notebook used this exact definition and the same period ordering. If the earlier penalty function defines adjacency differently, reconcile the definition before reporting results.

In [6]:
def adjacent_exam_penalty(schedule, enrolment_pairs):
    needed = {"exam_id", "period_index", "date"}
    if not needed.issubset(schedule.columns):
        raise ValueError(f"Schedule must contain {sorted(needed)}; found {schedule.columns.tolist()}")
    exam_info = schedule[["exam_id", "period_index", "date"]].copy()
    exam_info["exam_id"] = exam_info.exam_id.astype(str).str.strip()
    exam_info["period_index"] = pd.to_numeric(exam_info.period_index, errors="raise").astype(int)
    exam_info["date"] = exam_info.date.astype(str).str.strip()
    joined = enrolment_pairs.merge(exam_info, on="exam_id", how="inner")
    total = 0
    for _, group in joined.groupby("student_id", sort=False):
        group = group.sort_values(["date", "period_index"])
        rows = list(group[["date", "period_index"]].itertuples(index=False, name=None))
        for (date1, p1), (date2, p2) in zip(rows, rows[1:]):
            if date1 == date2 and int(p2) - int(p1) == 1:
                total += 1
    unmatched_exam_ids = enrolment_pairs.loc[~enrolment_pairs.exam_id.isin(exam_info.exam_id), "exam_id"].nunique()
    return int(total), int(unmatched_exam_ids), int(len(joined))

adaptive_penalty, adaptive_unmatched, adaptive_joined = adjacent_exam_penalty(adaptive_v3, enrolments)
cp_penalty, cp_unmatched, cp_joined = adjacent_exam_penalty(cp_sat, enrolments)
penalty_df = pd.DataFrame([
    {"Algorithm":"Adaptive v3", "Periods":adaptive_v3.period_index.nunique(), "Penalty":adaptive_penalty, "Matched enrolment rows":adaptive_joined, "Unmatched exam IDs":adaptive_unmatched},
    {"Algorithm":"CP-SAT", "Periods":cp_sat.period_index.nunique(), "Penalty":cp_penalty, "Matched enrolment rows":cp_joined, "Unmatched exam IDs":cp_unmatched},
])
display(penalty_df)

,Algorithm,Periods,Penalty,Matched enrolment rows,Unmatched exam IDs
0,Adaptive v3,17,976,33997,0
1,CP-SAT,16,0,33997,0


## 7. Append CP-SAT to the existing comparison

In [7]:
# Reuse the existing results_df when it exists; otherwise start a clean comparison table.
if "results_df" in globals() and isinstance(results_df, pd.DataFrame):
    comparison = results_df.copy()
else:
    comparison = pd.DataFrame(columns=["Algorithm", "Periods", "Conflict Violations", "Same-Day Adjacent-Exam Penalty", "Penalty Reduction (%)"])

for col in ["Algorithm", "Periods", "Conflict Violations", "Same-Day Adjacent-Exam Penalty", "Penalty Reduction (%)"]:
    if col not in comparison.columns:
        comparison[col] = np.nan
comparison = comparison[~comparison.Algorithm.isin(["Adaptive v3", "CP-SAT"])].copy()
comparison = pd.concat([comparison, pd.DataFrame([
    {"Algorithm":"Adaptive v3", "Periods":int(adaptive_v3.period_index.nunique()), "Conflict Violations":int(adaptive_check["Graph conflict violations"]), "Same-Day Adjacent-Exam Penalty":adaptive_penalty, "Penalty Reduction (%)":np.nan},
    {"Algorithm":"CP-SAT", "Periods":int(cp_sat.period_index.nunique()), "Conflict Violations":int(cp_check["Graph conflict violations"]), "Same-Day Adjacent-Exam Penalty":cp_penalty, "Penalty Reduction (%)":np.nan},
])], ignore_index=True)

dsatur = comparison.loc[comparison.Algorithm.eq("DSATUR"), "Same-Day Adjacent-Exam Penalty"].dropna()
if not dsatur.empty and float(dsatur.iloc[0]) != 0:
    baseline = float(dsatur.iloc[0])
    mask = comparison["Same-Day Adjacent-Exam Penalty"].notna()
    comparison.loc[mask, "Penalty Reduction (%)"] = ((baseline - comparison.loc[mask, "Same-Day Adjacent-Exam Penalty"]) / baseline * 100).round(2)
comparison = comparison.sort_values(["Periods", "Same-Day Adjacent-Exam Penalty"], na_position="last").reset_index(drop=True)
display(comparison)

,Algorithm,Periods,Conflict Violations,Same-Day Adjacent-Exam Penalty,Penalty Reduction (%)
0,CP-SAT,16,0,0,NaN
1,Adaptive v3,17,0,976,NaN


## 8. Export results

In [8]:
comparison_path = PROCESSED / "final_algorithm_comparison.csv"
validation_path = PROCESSED / "final_schedule_validation.csv"
penalty_path = PROCESSED / "final_penalty_comparison.csv"
comparison.to_csv(comparison_path, index=False)
validation_df.to_csv(validation_path, index=False)
penalty_df.to_csv(penalty_path, index=False)
print("Saved outputs:")
print(comparison_path)
print(validation_path)
print(penalty_path)

Saved outputs:
C:\Users\Nethmi\Documents\GitHub\adaptive-exam-timetabling\data\processed\final_algorithm_comparison.csv
C:\Users\Nethmi\Documents\GitHub\adaptive-exam-timetabling\data\processed\final_schedule_validation.csv
C:\Users\Nethmi\Documents\GitHub\adaptive-exam-timetabling\data\processed\final_penalty_comparison.csv


## 9. Interpretation and limitations

- CP-SAT currently has a feasible schedule using 16 periods and zero graph-conflict violations, according to the saved schedule and current graph.
- `FEASIBLE` does not prove 16 is the minimum. The solver must return `OPTIMAL` for the modeled objective to prove optimality.
- Compare penalty values with the earlier Adaptive v3 and DSATUR values only after confirming the penalty formula and period ordering are identical.
- This notebook does not certify complete Nottingham benchmark compliance for room capacities, room availability, fixed-room rules, earliness priorities, or constraints not explicitly modeled and checked.
- Do not claim full benchmark compliance until all applicable benchmark rules have been implemented and validated.

## Validate and Standardize the Adjacent-Exam Penalty

Both algorithms must use the same student–exam enrolments, exam-period mapping,
date mapping, and penalty definition. We will independently recompute the metric
and compare it with the existing results.

In [9]:
from pathlib import Path
import pandas as pd

# Find the project root (the notebook is expected to be inside src/)
cwd = Path.cwd()
if (cwd / "data" / "processed").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data" / "processed").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Cannot find data/processed. Run this notebook from the project "
        "root or its src folder."
    )

PROCESSED = PROJECT_ROOT / "data" / "processed"
RAW = PROJECT_ROOT / "data" / "raw" / "Nott"

# Load both saved schedules
adaptive_schedule = pd.read_csv(
    PROCESSED / "adaptive_v3_schedule.csv"
)
cp_sat_schedule = pd.read_csv(
    PROCESSED / "cp_sat_schedule.csv"
)

# Load the raw student-exam enrolment pairs
enrolments_df = pd.read_csv(
    RAW / "enrolements",
    sep=r"\s+",
    header=None,
    names=["student_id", "exam_id"],
    dtype=str
)

print("Adaptive v3:", adaptive_schedule.shape)
print("CP-SAT:", cp_sat_schedule.shape)
print("Enrolments:", enrolments_df.shape)
print("Adaptive columns:", adaptive_schedule.columns.tolist())
print("CP-SAT columns:", cp_sat_schedule.columns.tolist())
print("Enrolment columns:", enrolments_df.columns.tolist())

display(adaptive_schedule.head())
display(cp_sat_schedule.head())
display(enrolments_df.head())

Adaptive v3: (800, 4)
CP-SAT: (800, 5)
Enrolments: (33997, 2)
Adaptive columns: ['exam_id', 'period_index', 'date', 'time']
CP-SAT columns: ['exam_id', 'period_index', 'date', 'day', 'time']
Enrolment columns: ['student_id', 'exam_id']


,exam_id,period_index,date,time
0,LKA1FAE1,12,1995-01-27,09:00
1,LK10SAE1,2,1995-01-23,16:30
2,L81122E1,5,1995-01-24,16:30
3,N1A110E1,9,1995-01-26,09:00
4,C81ICAE1,6,1995-01-25,09:00


,exam_id,period_index,date,day,time
0,AAA033E1,0,1995-01-23,Monday,09:00
1,B31105E1,0,1995-01-23,Monday,09:00
2,C11101E1,0,1995-01-23,Monday,09:00
3,C12328E1,0,1995-01-23,Monday,09:00
4,C13567E1,0,1995-01-23,Monday,09:00


,student_id,exam_id
0,A890186790,R13001E1
1,A890186790,R13006E1
2,A890186790,R13016E1
3,A890186790,R13021E1
4,A890186790,R13022E1


In [10]:
from itertools import combinations
import pandas as pd

def calculate_adjacent_exam_penalty(schedule, enrolments):
    # Keep only the columns needed for this metric
    s = schedule[["exam_id", "period_index", "date"]].copy()
    e = enrolments[["student_id", "exam_id"]].copy()

    s["period_index"] = pd.to_numeric(s["period_index"], errors="raise")
    s["date"] = pd.to_datetime(s["date"]).dt.date

    if s["exam_id"].duplicated().any():
        raise ValueError("Duplicate exam IDs found in schedule.")

    # Attach each enrolled exam's scheduled period and date
    merged = e.merge(
        s, on="exam_id", how="left", validate="many_to_one",
        indicator=True
    )

    unmatched = int((merged["_merge"] != "both").sum())
    if unmatched:
        raise ValueError(f"{unmatched} enrolment records have unscheduled exams.")

    merged = merged.drop(columns="_merge")

    penalty = 0
    for _, group in merged.groupby("student_id", sort=False):
        # Avoid counting duplicate enrolment records for the same exam
        group = group.drop_duplicates("exam_id")
        records = group[["period_index", "date"]].to_dict("records")

        for a, b in combinations(records, 2):
            if (
                a["date"] == b["date"]
                and abs(a["period_index"] - b["period_index"]) == 1
            ):
                penalty += 1

    return penalty

penalty_check = pd.DataFrame([
    {
        "Algorithm": "Adaptive v3",
        "Periods": adaptive_schedule["period_index"].nunique(),
        "Penalty": calculate_adjacent_exam_penalty(
            adaptive_schedule, enrolments_df
        )
    },
    {
        "Algorithm": "CP-SAT",
        "Periods": cp_sat_schedule["period_index"].nunique(),
        "Penalty": calculate_adjacent_exam_penalty(
            cp_sat_schedule, enrolments_df
        )
    }
])

baseline = penalty_check.loc[
    penalty_check["Algorithm"] == "CP-SAT", "Penalty"
].iloc[0]

penalty_check["Penalty reduction vs CP-SAT (%)"] = (
    (baseline - penalty_check["Penalty"]) / baseline * 100
)

display(penalty_check)

,Algorithm,Periods,Penalty,Penalty reduction vs CP-SAT (%)
0,Adaptive v3,17,995,-inf
1,CP-SAT,16,0,NaN


In [11]:
# Save the current penalty comparison
output_path = PROCESSED / "final_penalty_comparison.csv"
penalty_check.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
display(penalty_check)

Saved: c:\Users\Nethmi\Documents\GitHub\adaptive-exam-timetabling\data\processed\final_penalty_comparison.csv


,Algorithm,Periods,Penalty,Penalty reduction vs CP-SAT (%)
0,Adaptive v3,17,995,-inf
1,CP-SAT,16,0,NaN


In [12]:
import pickle
import networkx as nx
import pandas as pd

# Load the conflict graph
graph_path = PROCESSED / "conflict_graph.gpickle"

try:
    conflict_graph = nx.read_gpickle(graph_path)
except AttributeError:
    with open(graph_path, "rb") as f:
        conflict_graph = pickle.load(f)

def validate_schedule(schedule, graph, algorithm):
    schedule = schedule.copy()
    schedule["exam_id"] = schedule["exam_id"].astype(str)
    schedule["period_index"] = pd.to_numeric(
        schedule["period_index"], errors="raise"
    )

    scheduled_exams = set(schedule["exam_id"])
    graph_exams = set(map(str, graph.nodes()))

    # Count each conflicting edge whose exams share a period
    period_map = schedule.set_index("exam_id")["period_index"].to_dict()
    conflict_violations = sum(
        1 for u, v in graph.edges()
        if str(u) in period_map
        and str(v) in period_map
        and period_map[str(u)] == period_map[str(v)]
    )

    return {
        "Algorithm": algorithm,
        "Graph exams": len(graph_exams),
        "Schedule rows": len(schedule),
        "Unique scheduled exams": schedule["exam_id"].nunique(),
        "Missing graph exams": len(graph_exams - scheduled_exams),
        "Extra exams": len(scheduled_exams - graph_exams),
        "Duplicate exam rows": int(schedule["exam_id"].duplicated().sum()),
        "Periods used": schedule["period_index"].nunique(),
        "Graph conflict violations": conflict_violations,
        "Graph feasible": conflict_violations == 0
            and graph_exams == scheduled_exams
            and not schedule["exam_id"].duplicated().any()
    }

validation_report = pd.DataFrame([
    validate_schedule(adaptive_schedule, conflict_graph, "Adaptive v3"),
    validate_schedule(cp_sat_schedule, conflict_graph, "CP-SAT")
])

display(validation_report)

# Save the report
validation_path = PROCESSED / "final_schedule_validation.csv"
validation_report.to_csv(validation_path, index=False)
print(f"Saved validation report: {validation_path}")

,Algorithm,Graph exams,Schedule rows,Unique scheduled exams,Missing graph exams,Extra exams,Duplicate exam rows,Periods used,Graph conflict violations,Graph feasible
0,Adaptive v3,800,800,800,0,0,0,17,0,True
1,CP-SAT,800,800,800,0,0,0,16,0,True


Saved validation report: c:\Users\Nethmi\Documents\GitHub\adaptive-exam-timetabling\data\processed\final_schedule_validation.csv


In [13]:
from pathlib import Path

RAW = PROJECT_ROOT / "data" / "raw" / "Nott"

print("Benchmark files:")
for path in sorted(RAW.iterdir()):
    if path.is_file():
        print(f"  {path.name} ({path.stat().st_size:,} bytes)")

readme_path = RAW / "README"

if readme_path.exists():
    print("\n--- Benchmark README ---")
    print(readme_path.read_text(encoding="utf-8", errors="replace"))
else:
    print("\nNo README file found.")

Benchmark files:
  data (3,770 bytes)
  enrolements (679,940 bytes)
  exams (46,400 bytes)
  README (950 bytes)
  students (126,336 bytes)

--- Benchmark README ---
This directory contains the main exam timetabling data for
the University of Nottingham, semester 1, 1994-95.
There are 800 exams provided by 46 departments, and 7896 students on 223
courses.  There are 33997 enrolements, so each student is sitting an average
of 4.3 exams, and each exam is sat by an average of 42.5 students.

The data is in the following files:

students      the list of students, with their courses;

                 student code      10 characters
                 course code        4     "

exams         the list of exams, containing the following fields;

                 exam code          8 characters
                 description       40     "
                 duration           4     "
                 department code    2     "

enrolements   a list of exams for each student;

                 stud

In [14]:
# Inspect the benchmark's additional constraints file
data_path = RAW / "data"

print("Benchmark data file contents:\n")
print(data_path.read_text(encoding="utf-8", errors="replace"))

Benchmark data file contents:

DATES
-----
Mon 23rd Jan - Sat 4th Feb 1995

TIMES
-----
Mon - Fri  9:00 (3hrs), 13:30 (2hrs), 16:30 (2hrs)
Sat        9:00 (3hrs)

ROOMS
-----
TRENT-HALL   125
TRENT-L19     80
TRENT-B46     40
COPSE-2       40
COPSE-3       50
COPSE-4       35
COPSE-5       30
COPSE-6       25
PRTLND-GRND  200
POPE-A13      80 \ together  \
POPE-A14      80 /           |  Rooms that are together can be combined,
ART-LECTURE   80 \ together  -  ie. used as one room.  This is not as good as
ART-SEMINAR   15 /           |  using one room, but is better than splitting
SPORT-LGE1   250 \ together  |  an exam over two distant rooms.
SPORT-LGE2   230 /           /
SPORT-SMALL  270

ROOM ASSIGNMENTS
----------------
V41103E1   ART-SEMINAR
V41112E1   ART-LECTURE
V41124E1   ART-SEMINAR
V41133E1   ART-LECTURE
V41142E1   ART-LECTURE
V41143E1   ART-SEMINAR
V42103E1   ART-LECTURE
V42112E1   ART-LECTURE
V42117E1   ART-LECTURE
V42124E1   ART-LECTURE
V42133E1   ART-LECTURE
V42143E1   AR

In [15]:
import pandas as pd
from itertools import combinations

# The benchmark permits weekdays at 09:00, 13:30 and 16:30,
# plus Saturdays at 09:00, within 23 Jan–4 Feb 1995.
ALLOWED_WEEKDAY_TIMES = {"09:00", "13:30", "16:30"}
ALLOWED_SATURDAY_TIMES = {"09:00"}
START_DATE = pd.Timestamp("1995-01-23").date()
END_DATE = pd.Timestamp("1995-02-04").date()

# Explicit date restrictions from the benchmark data file.
FIXED_DATES = {
    "F321Q6E1": {"1995-01-27"},
    "F321T6E1": {"1995-01-30"},
}
DATE_RANGES = {
    "H21M01E1": ("1995-01-23", "1995-01-24"),
    "H22M02E1": ("1995-01-23", "1995-01-24"),
    "H2CM04E1": ("1995-01-23", "1995-01-24"),
}
BEFORE_DATES = {
    "G13RE2E1": "1995-01-30",
}

# Time-slot restrictions from the benchmark.
AM_ONLY = {"K1AHWAE2", "H63122E1"}
THURSDAY_PM_ONLY = {"V13101E1"}

# Precedence: first exam must be earlier than second.
PRECEDENCE_PAIRS = [
    ("F13P03E1", "F13X03E1"),
    ("F13P05E1", "F13X04E1"),
]

# Immediate succession: the second exam must be in the next period.
IMMEDIATE_PAIRS = [
    ("H3BFM2E1", "H3BFM2E2"),
]

def validate_benchmark_rules(schedule, enrolments, algorithm):
    s = schedule.copy()
    s["exam_id"] = s["exam_id"].astype(str)
    s["date"] = pd.to_datetime(s["date"]).dt.date
    s["time"] = s["time"].astype(str).str.strip().str[:5]
    s["period_index"] = pd.to_numeric(s["period_index"], errors="raise")
    by_exam = s.set_index("exam_id").to_dict("index")

    results = []

    def record(rule, violations, detail):
        results.append({
            "Algorithm": algorithm,
            "Constraint": rule,
            "Violations": len(violations),
            "Examples": "; ".join(map(str, violations[:8])),
            "Detail": detail,
        })

    # Calendar date/time validity
    invalid_calendar = []
    for exam, row in by_exam.items():
        d, t = row["date"], row["time"]
        allowed = (
            START_DATE <= d <= END_DATE
            and (
                (d.weekday() < 5 and t in ALLOWED_WEEKDAY_TIMES)
                or (d.weekday() == 5 and t in ALLOWED_SATURDAY_TIMES)
            )
        )
        if not allowed:
            invalid_calendar.append(f"{exam} ({d} {t})")
    record("Allowed calendar slots", invalid_calendar,
           "Checks dates, weekdays/Saturday and start times.")

    # Fixed dates
    violations = []
    for exam, dates in FIXED_DATES.items():
        if exam in by_exam and str(by_exam[exam]["date"]) not in dates:
            violations.append(
                f"{exam} scheduled {by_exam[exam]['date']}"
            )
    record("Fixed dates", violations, "Explicit date requirements.")

    # Date ranges
    violations = []
    for exam, (lo, hi) in DATE_RANGES.items():
        if exam in by_exam:
            d = str(by_exam[exam]["date"])
            if not lo <= d <= hi:
                violations.append(f"{exam} scheduled {d}")
    record("Date-range restrictions", violations,
           "Exam must fall within its specified date range.")

    # Before a specified date
    violations = []
    for exam, deadline in BEFORE_DATES.items():
        if exam in by_exam and str(by_exam[exam]["date"]) >= deadline:
            violations.append(f"{exam} scheduled {by_exam[exam]['date']}")
    record("Before-date restrictions", violations,
           "Exam must occur before the specified date.")

    # Morning-only
    violations = []
    for exam in AM_ONLY:
        if exam in by_exam and by_exam[exam]["time"] != "09:00":
            violations.append(f"{exam} at {by_exam[exam]['time']}")
    record("Morning-only exams", violations,
           "09:00 is treated as the morning slot.")

    # Thursday afternoon for V13101E1
    violations = []
    exam = "V13101E1"
    if exam in by_exam:
        row = by_exam[exam]
        if row["date"].weekday() != 3 or row["time"] not in {"13:30", "16:30"}:
            violations.append(f"{exam} scheduled {row['date']} {row['time']}")
    record("Thursday afternoon restriction", violations,
           "Thursday is weekday index 3.")

    # Precedence pairs
    violations = []
    for first, second in PRECEDENCE_PAIRS:
        if first in by_exam and second in by_exam:
            a, b = by_exam[first], by_exam[second]
            if (a["date"], a["time"]) >= (b["date"], b["time"]):
                violations.append(f"{first} must precede {second}")
        else:
            violations.append(f"Missing exam: {first} or {second}")
    record("Precedence pairs", violations,
           "Checks chronological order, not room allocation.")

    # Immediate succession
    violations = []
    for first, second in IMMEDIATE_PAIRS:
        if first in by_exam and second in by_exam:
            a, b = by_exam[first], by_exam[second]
            if b["period_index"] != a["period_index"] + 1:
                violations.append(
                    f"{first} period {a['period_index']} -> "
                    f"{second} period {b['period_index']}"
                )
        else:
            violations.append(f"Missing exam: {first} or {second}")
    record("Immediate succession", violations,
           "Checks adjacent period indices.")

    # Student gap: benchmark requires at least one period between
    # same-day consecutive exams. Overnight gaps are acceptable.
    merged = enrolments[["student_id", "exam_id"]].merge(
        s[["exam_id", "period_index", "date"]],
        on="exam_id", how="left", validate="many_to_one"
    )
    missing = merged[merged["period_index"].isna()]
    record("Enrolment-to-schedule coverage",
           missing["exam_id"].astype(str).unique().tolist(),
           "Every enrolled exam must be scheduled.")

    gap_violations = []
    merged = merged.dropna(subset=["period_index"])
    for student, group in merged.groupby("student_id", sort=False):
        group = group.drop_duplicates("exam_id")
        # Same-day exam pairs in consecutive periods violate the gap rule.
        rows = group[["exam_id", "period_index", "date"]].to_dict("records")
        for a, b in combinations(rows, 2):
            if (
                a["date"] == b["date"]
                and abs(int(a["period_index"]) - int(b["period_index"])) == 1
            ):
                gap_violations.append(
                    f"{student}: {a['exam_id']} / {b['exam_id']}"
                )
    record("Student one-period gap", gap_violations,
           "Counts student-level pairs on the same date in adjacent periods.")

    return results

benchmark_checks = pd.DataFrame(
    validate_benchmark_rules(adaptive_schedule, enrolments_df, "Adaptive v3")
    + validate_benchmark_rules(cp_sat_schedule, enrolments_df, "CP-SAT")
)

display(benchmark_checks)

benchmark_checks.to_csv(
    PROCESSED / "benchmark_constraint_validation_partial.csv",
    index=False
)
print("Saved partial benchmark validation report.")

,Algorithm,Constraint,Violations,Examples,Detail
0,Adaptive v3,Allowed calendar slots,0,,"Checks dates, weekdays/Saturday and start times."
1,Adaptive v3,Fixed dates,2,F321Q6E1 scheduled 1995-01-23; F321T6E1 schedu...,Explicit date requirements.
2,Adaptive v3,Date-range restrictions,2,H21M01E1 scheduled 1995-01-30; H2CM04E1 schedu...,Exam must fall within its specified date range.
3,Adaptive v3,Before-date restrictions,0,,Exam must occur before the specified date.
4,Adaptive v3,Morning-only exams,0,,09:00 is treated as the morning slot.
5,Adaptive v3,Thursday afternoon restriction,1,V13101E1 scheduled 1995-01-25 09:00,Thursday is weekday index 3.
6,Adaptive v3,Precedence pairs,2,F13P03E1 must precede F13X03E1; F13P05E1 must ...,"Checks chronological order, not room allocation."
7,Adaptive v3,Immediate succession,1,H3BFM2E1 period 3 -> H3BFM2E2 period 12,Checks adjacent period indices.
8,Adaptive v3,Enrolment-to-schedule coverage,0,,Every enrolled exam must be scheduled.
9,Adaptive v3,Student one-period gap,995,A920097599: Q32302E1 / Q3A201E1; A920340635: V...,Counts student-level pairs on the same date in...


Saved partial benchmark validation report.


In [16]:
def summarize_student_gap_violations(schedule, enrolments, algorithm):
    s = schedule[["exam_id", "period_index", "date"]].copy()
    s["date"] = pd.to_datetime(s["date"]).dt.date
    s["period_index"] = pd.to_numeric(s["period_index"], errors="raise")

    merged = enrolments[["student_id", "exam_id"]].merge(
        s, on="exam_id", how="left", validate="many_to_one"
    )

    affected_students = set()
    violation_pairs = []

    for student, group in merged.groupby("student_id", sort=False):
        group = group.dropna(subset=["period_index"]).drop_duplicates("exam_id")
        records = group[["exam_id", "period_index", "date"]].to_dict("records")

        for i in range(len(records)):
            for j in range(i + 1, len(records)):
                a, b = records[i], records[j]

                if (
                    a["date"] == b["date"]
                    and abs(int(a["period_index"]) - int(b["period_index"])) == 1
                ):
                    affected_students.add(student)
                    violation_pairs.append({
                        "student_id": student,
                        "exam_1": a["exam_id"],
                        "period_1": a["period_index"],
                        "exam_2": b["exam_id"],
                        "period_2": b["period_index"],
                        "date": a["date"],
                    })

    return {
        "Algorithm": algorithm,
        "Affected students": len(affected_students),
        "Violating student-exam pairs": len(violation_pairs),
        "Examples": pd.DataFrame(violation_pairs).head(10),
    }

adaptive_gap = summarize_student_gap_violations(
    adaptive_schedule, enrolments_df, "Adaptive v3"
)
cpsat_gap = summarize_student_gap_violations(
    cp_sat_schedule, enrolments_df, "CP-SAT"
)

display(pd.DataFrame([
    {k: v for k, v in adaptive_gap.items() if k != "Examples"},
    {k: v for k, v in cpsat_gap.items() if k != "Examples"},
]))

print("Adaptive v3 examples:")
display(adaptive_gap["Examples"])

print("CP-SAT examples:")
display(cpsat_gap["Examples"])

,Algorithm,Affected students,Violating student-exam pairs
0,Adaptive v3,759,995
1,CP-SAT,0,0


Adaptive v3 examples:


,student_id,exam_1,period_1,exam_2,period_2,date
0,A920097599,Q32302E1,5,Q3A201E1,4,1995-01-24
1,A920340635,V13159E1,12,V6B329E1,13,1995-01-27
2,A920396285,AAA011E1,12,R22212E1,13,1995-01-27
3,A920574816,V62200E1,12,V6B329E1,13,1995-01-27
4,A920641200,M1B363E1,13,V13185E1,12,1995-01-27
5,A920722244,Q83184E1,12,V6B329E1,13,1995-01-27
6,A921289681,M1B363E1,13,M1B382E1,14,1995-01-27
7,A921388641,Q33307E1,2,V6A100E1,1,1995-01-23
8,A922287749,V72119E1,12,V7A204E1,13,1995-01-27
9,A922898006,F311X1E1,7,V72AINE1,6,1995-01-25


CP-SAT examples:


""


In [18]:
# ============================================================
# Additional benchmark validation: coincidence and different-time
# ============================================================

coincidence_groups = [
    ["LK44FAE1", "LK44GAE1", "LK44SAE1"],
    ["LKC3FAE1", "LKC3GAE1", "LKC3SAE1", "LKC3RAE1"],
    ["C81IAAE1", "C81J1AE2", "C81J4AE2", "C81HAAE1"],
    ["C81HBAE1", "C81IBAE1", "C81J1AE1", "C81J2AE1", "C81J3AE1"],
    ["C81HCAE1", "C81ICAE1", "C81J2AE2", "C81J4AE1", "C81J5AE1"],
    ["C81HDAE1", "C81IDAE1", "C81J3AE2", "C81J5AE2"],
    ["C81MJAE1", "C81MSAE1", "C82MCPE1"],
    ["C82BCSE1", "C82BUAE1"],
    ["C82MHAE1", "C82MJAE1"],
    ["R23102E1", "R23104E1"],
    ["F12I01E1", "F12X04E1", "F12X05E1"],
    ["F12O01E1", "F12X04E2"],
    ["F12P01E1", "F12P02E1", "F12P03E1", "F12X04E3"],
    ["F13I01E1", "F13I02E1", "F13X01E1"],
    ["F13O01E1", "F13O03E1"],
    ["F13X03E1", "F13X04E1"],
    ["M11341E1", "M11345E1"],
    ["M11343E1", "M11347E1"],
    ["M12353E1", "M13369E1"],
    ["H8A009E1", "H8A011E1"],
    ["Q31301E1", "Q3A301E1"],
    ["Q32302E1", "Q3A303E1"],
    ["HG2J3BE1", "HGCEMME1"],
    ["HG3J5AE1", "HG3J5BE1"],
    ["HGCEMAE1", "HGCEMBE1", "HGCEMCE1"],
    ["G5P002E1", "C8CXESE1"],
    ["L31073E1", "L31107E1"],
    ["V7A101E1", "V7A117E1"],
    ["L11101E1", "L11102E1"],
    ["L11110E1", "L11111E1"],
    ["L11130E1", "L11140E1"],
    ["M11351E1", "M11451E1"],
]

different_time_pairs = [
    ("H8B040E1", "H8C001E1"),
]

def validate_extra_constraints(schedule, algorithm):
    period_by_exam = dict(
        zip(schedule["exam_id"].astype(str), schedule["period_index"].astype(int))
    )

    results = []

    # Coincidence: all listed exams must occupy the same period.
    for group in coincidence_groups:
        present = [exam for exam in group if exam in period_by_exam]
        missing = [exam for exam in group if exam not in period_by_exam]

        if missing:
            results.append({
                "Algorithm": algorithm,
                "Constraint": "Coincidence group",
                "Exams": ", ".join(group),
                "Status": "INCOMPLETE — missing exam(s): " + ", ".join(missing),
            })
        elif len({period_by_exam[exam] for exam in present}) == 1:
            results.append({
                "Algorithm": algorithm,
                "Constraint": "Coincidence group",
                "Exams": ", ".join(group),
                "Status": "PASS",
            })
        else:
            results.append({
                "Algorithm": algorithm,
                "Constraint": "Coincidence group",
                "Exams": ", ".join(group),
                "Status": "FAIL",
            })

    # Different-time rule: the two exams must not occupy the same period.
    for exam_a, exam_b in different_time_pairs:
        if exam_a not in period_by_exam or exam_b not in period_by_exam:
            status = "INCOMPLETE — exam missing"
        elif period_by_exam[exam_a] != period_by_exam[exam_b]:
            status = "PASS"
        else:
            status = "FAIL"

        results.append({
            "Algorithm": algorithm,
            "Constraint": "Different-time pair",
            "Exams": f"{exam_a}, {exam_b}",
            "Status": status,
        })

    return results


extra_results = []
extra_results += validate_extra_constraints(adaptive_v3, "Adaptive v3")
extra_results += validate_extra_constraints(cp_sat, "CP-SAT")

extra_validation = pd.DataFrame(extra_results)

display(
    extra_validation.groupby(
        ["Algorithm", "Constraint", "Status"], dropna=False
    ).size().reset_index(name="Count")
)

display(extra_validation[extra_validation["Status"] != "PASS"])

,Algorithm,Constraint,Status,Count
0,Adaptive v3,Coincidence group,FAIL,28
1,Adaptive v3,Coincidence group,PASS,4
2,Adaptive v3,Different-time pair,PASS,1
3,CP-SAT,Coincidence group,FAIL,2
4,CP-SAT,Coincidence group,PASS,30
5,CP-SAT,Different-time pair,PASS,1


,Algorithm,Constraint,Exams,Status
0,Adaptive v3,Coincidence group,"LK44FAE1, LK44GAE1, LK44SAE1",FAIL
1,Adaptive v3,Coincidence group,"LKC3FAE1, LKC3GAE1, LKC3SAE1, LKC3RAE1",FAIL
2,Adaptive v3,Coincidence group,"C81IAAE1, C81J1AE2, C81J4AE2, C81HAAE1",FAIL
3,Adaptive v3,Coincidence group,"C81HBAE1, C81IBAE1, C81J1AE1, C81J2AE1, C81J3AE1",FAIL
4,Adaptive v3,Coincidence group,"C81HCAE1, C81ICAE1, C81J2AE2, C81J4AE1, C81J5AE1",FAIL
5,Adaptive v3,Coincidence group,"C81HDAE1, C81IDAE1, C81J3AE2, C81J5AE2",FAIL
6,Adaptive v3,Coincidence group,"C81MJAE1, C81MSAE1, C82MCPE1",FAIL
7,Adaptive v3,Coincidence group,"C82BCSE1, C82BUAE1",FAIL
8,Adaptive v3,Coincidence group,"C82MHAE1, C82MJAE1",FAIL
10,Adaptive v3,Coincidence group,"F12I01E1, F12X04E1, F12X05E1",FAIL
